# Sentiment Analysis with HuggingFace and PySpark

This is a small personal project. I take a set of car reviews, use a ready-made
HuggingFace model to predict whether each review is positive or negative, and use
PySpark to handle the data. At the end I check how accurate the model is.

**Steps**
1. Install the libraries
2. Start Spark
3. Load the reviews
4. Load the sentiment model
5. Predict sentiment for each review
6. Compare predictions with the real labels (accuracy and F1)


## 1. Install the libraries
Run this cell the first time only.


In [3]:
!pip install pyspark transformers torch scikit-learn


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 450.1/450.1 MB 25.9 MB/s  0:00:10:00:0100:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 27.8 MB/s  0:00:006m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.4/780.4 kB 28.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.4/4.4 MB 39.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 27.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 526.6/526.6 MB 17.8 MB/s  0:00:15m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.2/366.2 MB 27.0 MB/s  0:00:10m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170.1 MB 39.8 MB/s  0:00:04m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.0/206.0 MB 35.2 MB/s  0:00:05m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 45.8 MB/s  0:00:016m0:00:0100:01
   ━━━━━━

## 2. Start Spark


In [4]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("SentimentAnalysis").getOrCreate()


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/07/31 08:59:33 WARN Utils: Your hostname, codespaces-fe6721, resolves to a loopback address: 127.0.0.1; using 10.0.3.206 instead (on interface eth0)
26/07/31 08:59:33 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/31 08:59:35 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## 3. Load the car reviews
The file is separated by semicolons, so I tell Spark that.


In [5]:
df = spark.read.option("header", True).option("sep", ";").csv("car_reviews.csv")
df.show(5)


+--------------------+--------------------+
|              Review|               Class|
+--------------------+--------------------+
|I am very satisfi...|            POSITIVE|
|The car is fine. ...|            NEGATIVE|
|My first foreign ...|            POSITIVE|
|"I've come across...| my 2006 Pathfind...|
|I've been dreamin...|            POSITIVE|
+--------------------+--------------------+



## 4. Load the sentiment model
This downloads a small pre-trained model from HuggingFace the first time.


In [6]:
from transformers import pipeline

classifier = pipeline("sentiment-analysis",
                      model="distilbert/distilbert-base-uncased-finetuned-sst-2-english")


/home/codespace/.python/current/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 104/104 [00:00<00:00, 3015.67it/s]


## 5. Predict the sentiment for each review
I make a small function and apply it to the Review column with a UDF.


In [7]:
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

def get_sentiment(review):
    result = classifier(review[:512])   # cut long reviews so the model doesn't complain
    return result[0]["label"]

sentiment_udf = udf(get_sentiment, StringType())

df = df.withColumn("Predicted", sentiment_udf(df["Review"]))
df.select("Class", "Predicted", "Review").show(10, truncate=60)


/home/codespace/.python/current/lib/python3.12/site-packages/pyspark/sql/udf.py:120: RuntimeWarning: Arrow optimization failed to enable because PyArrow or Pandas is not installed. Falling back to a non-Arrow-optimized UDF.
  warnings.warn(


+------------------------------------------------------------+---------+------------------------------------------------------------+
|                                                       Class|Predicted|                                                      Review|
+------------------------------------------------------------+---------+------------------------------------------------------------+
|                                                    POSITIVE| POSITIVE|I am very satisfied with my 2014 Nissan NV SL. I use this...|
|                                                    NEGATIVE| POSITIVE|The car is fine. It's a bit loud and not very powerful. O...|
|                                                    POSITIVE| POSITIVE|         My first foreign car. Love it, I would buy another.|
| my 2006 Pathfinder had a smoother ride! I would rate thi...| NEGATIVE|"I've come across numerous reviews praising the Rogue, an...|
|                                                    POSITIVE|

## 6. Turn the labels into numbers
POSITIVE = 1, NEGATIVE = 0 so I can compare them.


In [8]:
from pyspark.sql.functions import when

df = df.withColumn("actual", when(df["Class"] == "POSITIVE", 1).otherwise(0))
df = df.withColumn("predicted", when(df["Predicted"] == "POSITIVE", 1).otherwise(0))
df.select("actual", "predicted").show()


+------+---------+
|actual|predicted|
+------+---------+
|     1|        1|
|     0|        1|
|     1|        1|
|     0|        0|
|     1|        0|
+------+---------+



## 7. Check accuracy and F1 score


In [9]:
rows = df.select("actual", "predicted").collect()
actual = [r["actual"] for r in rows]
predicted = [r["predicted"] for r in rows]

from sklearn.metrics import accuracy_score, f1_score

print("Accuracy:", accuracy_score(actual, predicted))
print("F1 score:", f1_score(actual, predicted))


Accuracy: 0.6
F1 score: 0.6666666666666666


## 8. See which reviews the model got wrong


In [10]:
df.filter(df["actual"] != df["predicted"]).select("Class", "Predicted", "Review").show(truncate=80)


+--------+---------+--------------------------------------------------------------------------------+
|   Class|Predicted|                                                                          Review|
+--------+---------+--------------------------------------------------------------------------------+
|NEGATIVE|        1|The car is fine. It's a bit loud and not very powerful. On one hand, compared...|
|POSITIVE|        0|I've been dreaming of owning an SUV for quite a while, but I've been driving ...|
+--------+---------+--------------------------------------------------------------------------------+



In [11]:
spark.stop()
